# Leakage safe regression and classification pipeline
# using cross-validation compare models , tune selected hyperparameters

Hyperparameter - things used to prevent overfitting and underfitting , eg - regularization strength

model parameter - weight and bias

# Interpret fold variability and justify final model



# EXERCISE A - CLASSIFICATION

exclude ticket_id , target - escalated - target
stratified train / test split - means the target in both include both yes and no

pipeline - median imputation and scaling numeric columns 
- most frequent imputation and one-hot encoding for categorical columns


models - logistic regression , k-NN , decision tree using 5-fold stratified k fold
F1 primary metric
mean and standard deviation - from the training data


Tuning >= 1 candidate using Grid / Randomized search.
Confusion matrix

In [ ]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.tree import DecisionTreeClassifier


from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import f1_score, precision_score, recall_score



# model 
from sklearn.linear_model import LogisticRegression


# loading and separating test and train data
df = pd.read_csv("./data/data/classification_escalation.csv")

X = df.drop(columns=["ticket_id", "escalated"])
y = df["escalated"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# pre-processing the data 
# enter the median for numerical values
# enter the most frequent value for categorical values

#  split the data into 5 folds -> scale -> impute -> train -> validate
# this is the pipeline to remove data leakage 

num_cols = ["complexity_score", "team_experience_years", "customer_tenure_months"]
cat_cols = ["priority", "channel"]

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
])


# first we have to do cross validation split the training data into 5 

# logistic regression model


# median will affect the data so after spliting is when u shld impute


log_reg_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# fit the model at all folds
f1_scores = cross_val_score(log_reg_pipe, X_train, y_train, cv=cv, scoring="f1")


print(f"F1 scores for each fold: {f1_scores}")

print(f"Mean F1: {f1_scores.mean():.4f}")
print(f"Std F1:  {f1_scores.std():.4f}")



# precision - of all the positive which one are true positive
# recall - of all the true positive how many did we catch

# F1 combines both


# KNN classifier




knn_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", KNeighborsClassifier(n_neighbors=5))
])



cv2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
knn_f1_scores = cross_val_score(knn_pipe, X_train, y_train, cv=cv2, scoring="f1")

print(f"F1 scores for each fold in knn: {knn_f1_scores}")
print(f"Mean F1: {knn_f1_scores.mean():.4f}")
print(f"Std F1:  {knn_f1_scores.std():.4f}")



# Decision tree classifier

dt_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", DecisionTreeClassifier(random_state=42)),
])



cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
dt_f1_scores = cross_val_score(dt_pipe, X_train, y_train, cv=cv, scoring="f1")

print(f"F1 scores for each fold for the decision tree: {dt_f1_scores}")
print(f"Mean F1: {dt_f1_scores.mean():.4f}")
print(f"Std F1:  {dt_f1_scores.std():.4f}")




lr_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

# I chose GridSearch CV for hyperparameter tuning bc I will have limited number of values for the hyper parameters
# the hyperparameters - model_c , model_classw_weight , model_solver
lr_grid = {
    # regularization strength, smaller values specify stronger regularization
    "model__C": [0.01, 0.1, 1.0, 10.0, 100.0],
    # class weights
    "model__class_weight": [None, "balanced"],
    # solver for optimization
    "model__solver": ["lbfgs", "liblinear"],
}


lr_search = GridSearchCV(lr_pipe, lr_grid, cv=cv, scoring="f1", n_jobs=-1, refit=True)
lr_search.fit(X_train, y_train)

knn_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", KNeighborsClassifier()),
])


# number of neighbours to watch out 
# model_weights , model_metric
knn_grid = {
    "model__n_neighbors": [3, 5, 7, 11, 15],
    "model__weights": ["uniform", "distance"],
    "model__metric": ["euclidean", "manhattan"],
}
knn_search = GridSearchCV(knn_pipe, knn_grid, cv=cv, scoring="f1", n_jobs=-1, refit=True)
knn_search.fit(X_train, y_train)

dt_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", DecisionTreeClassifier(random_state=42)),
])


# for decision tree 
dt_grid = {
    "model__max_depth": [2, 3, 4, 5, 8, None],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 3, 5, 10],
    "model__class_weight": [None, "balanced"],
}
dt_search = GridSearchCV(dt_pipe, dt_grid, cv=cv, scoring="f1", n_jobs=-1, refit=True)
dt_search.fit(X_train, y_train)

print("Best LR:", lr_search.best_params_, f"CV F1: {lr_search.best_score_:.4f}")
print("Best k-NN:", knn_search.best_params_, f"CV F1: {knn_search.best_score_:.4f}")
print("Best DT:", dt_search.best_params_, f"CV F1: {dt_search.best_score_:.4f}")





F1 scores for each fold: [0.39130435 0.3255814  0.36363636 0.37209302 0.5106383 ]
Mean F1: 0.3927
Std F1:  0.0627
F1 scores for each fold in knn: [0.36734694 0.3255814  0.42553191 0.28571429 0.4       ]
Mean F1: 0.3608
Std F1:  0.0503
F1 scores for each fold for the decision tree: [0.4        0.2295082  0.32258065 0.37288136 0.32835821]
Mean F1: 0.3307
Std F1:  0.0581
Best LR: {'model__C': 0.1, 'model__class_weight': 'balanced', 'model__solver': 'liblinear'} CV F1: 0.5212
Best k-NN: {'model__metric': 'euclidean', 'model__n_neighbors': 3, 'model__weights': 'uniform'} CV F1: 0.3906
Best DT: {'model__class_weight': 'balanced', 'model__max_depth': 2, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2} CV F1: 0.4895


GridSearchCV

hyperparameter tuning 

# Linear Regression , Ridge regression and decision tree regression



In [16]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

import numpy as np

# 1. Load data, exclude ticket_id, and split into train/test
df = pd.read_csv("./data/data/resolution_time_regression.csv")
X = df.drop(columns=["ticket_id", "resolution_time_hours"])
y = df["resolution_time_hours"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Preprocessing pipeline
num_cols = ["complexity_score", "team_experience_years", "customer_tenure_months"]
cat_cols = ["priority", "channel"]

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
])

# 3. Compare Linear Regression, Ridge Regression, and Decision Tree using 5-fold CV & MAE
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0, random_state=42),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=42),
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", model),
    ])
    mae_scores = -cross_val_score(
        pipe, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error"
    )
    pipe.fit(X_train, y_train)
    test_mae = mean_absolute_error(y_test, pipe.predict(X_test))

    print(f"{name}:")
    print(f"  5-Fold CV Mean MAE: {mae_scores.mean():.4f} hrs (Std: {mae_scores.std():.4f})")
    print(f"  Test MAE:           {test_mae:.4f} hrs\n")



# tuning the alpha for ridge regression
# u gotta scale the data before tuning the alpha for ridge regression

ridge_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", Ridge(random_state=42)),
    # bc alpha is the hyper parameter being tuned
])

grid = {"model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]}


search = GridSearchCV(ridge_pipe, grid, cv=cv, scoring="neg_mean_absolute_error", refit=True)
search.fit(X_train, y_train)


best_idx = search.best_index_
fold_maes = np.array([
    -search.cv_results_[f"split{i}_test_score"][best_idx]
    for i in range(cv.n_splits)
])

print("fold maes")
print(fold_maes)

print("Tuned Ridge Regression")
print(f"Best alpha: {search.best_params_['model__alpha']}")
print(f"CV Mean MAE: {fold_maes.mean():.4f} hrs | CV Std MAE: {fold_maes.std():.4f} hrs")

# 5. Evaluate the selected model ONCE on the held-out test set
y_test_pred = search.predict(X_test)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = root_mean_squared_error(y_test, y_test_pred)

print("Final Test Set Evaluation (Single Run)")
print(f"Test MAE:  {test_mae:.4f} hrs")
print(f"Test RMSE: {test_rmse:.4f} hrs")

Linear Regression:
  5-Fold CV Mean MAE: 2.0351 hrs (Std: 0.1717)
  Test MAE:           2.2300 hrs

Ridge Regression:
  5-Fold CV Mean MAE: 2.0351 hrs (Std: 0.1714)
  Test MAE:           2.2310 hrs

Decision Tree Regressor:
  5-Fold CV Mean MAE: 3.2391 hrs (Std: 0.0538)
  Test MAE:           3.2354 hrs

fold maes
[2.34176264 1.81704617 1.96533874 2.04371996 2.00772292]
Tuned Ridge Regression
Best alpha: 0.1
CV Mean MAE: 2.0351 hrs | CV Std MAE: 0.1716 hrs
Final Test Set Evaluation (Single Run)
Test MAE:  2.2301 hrs
Test RMSE: 3.4028 hrs


# Explain outliers and why MAE and RMSE may prefer different models. 


Outliers in a data are values that lie away from the normal cluster and then impact the MAE and RMSE. But if our model predicts good for normal values but in the outliers its guess is terrible or our model is sloppy for normal values but good for outlier values the MAE and RMSE will behave differently. Since MAE is just the sum of absolute value of the errors even though the difference from the outliers is huge it will simply make it positive and include it in the sum , but RMSE will square it and put it in the sum , thus even though at the end we put it in the square root outliers higly impact the RMSE values for our system. Thus MAE will be fogiving for models that do bad guessing for outliers and RMSE prefers models that can make good guesses towards outliers.

# The final Report

The thing I have been doing here is trianing 2 types of models , classification and regression models with different outputs. The classification model tries to check if the task escalates or not while the regression model tries to predict how much time will be required to solve the task at hand. Since the data contains null values for both numeric and categorical values and also categorical values are put here in words we need to have a pre processing pipeline , that prevents leakage i.e the impact of testing data on the training models . Thus in the preprocessing part what I did was to separate the data into training and testing sets , then the training dataset  into 5  folds for cross-validation , then impute the value from the training parts in the cross-validation set and then scale the data using those values and one-hot encoding to make sure the model sees 0 or 1 and not 1 , 2 , 3 .. for categorical values to spare it from being confused and thinking the categorical column with value 3 has got more importance . Then I started using different models for training like logistic regression , KNN neighbours and decision tree for classification models and linear regression, ridge regression and decision tree regression for regression models. Using different metrics to select the best model using cross-validation and picking the highest F1 value for classification models and the lowest MAE for regresion models. F1 is the best describer of the model because high value of F1 mean the model's prediction for positive will be correct for the highest amount and MAE will mean when I test the model it has got low difference between test and predicted values. To prevent leakage I have been doing the selection and pipeline on the training part of the cross-validation folds.


# Why stratify the classification split? 

Stratifying will keep the result balance to be the same in the test and training set. This helps to train the model in the data that represents the whole model rather than just a piece of it.

# Why must preprocessing be inside the pipeline? 

Pipeline data preprocessing helps to prevent data-leakage , because everything we will be writing in the pipeline will get done on the cross-validation training set and will use the training set's data to process the testing set as well.

# What does fold variability tell you ?

Fold variability means when training the data in different number of splits how much variability is there in the MAE and different metric evaluation. So if there is high fold variability then it means my model is not good at generalizing because of its high difference. But low values of fold variability will mean higher generalization.

# Why should the test set be evaluated only once?

Testing is finally used for predicting the values so it is a data that my model hasn't been trained on and is new. I have no business of re-running my model on it because I am not using the test dataset for model or hyperparameter tuning , I am simply using it for checking how my already pre-tuned model works when faced with unseen data.

# What deployment constraints might change your choice?

One thing that will impact my choice of model is latency in deployed version. If the api requesting this service is synchronous then the model will have to answer quickly using linear regression and ridge regression rather than using slow models like decision trees that require high computation for deeper trees.



